In [0]:
import os
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

# Volume root and tracking file path
root_path = "/Volumes/0725catalog/src/data"
tracking_file = "/Volumes/0725catalog/src/data/processed_files.json"

# Recursively find all CSVs using Python os.walk
def find_all_csv_files(path):
    csv_files = []
    for dirpath, _, filenames in os.walk(path):
        for file in filenames:
            if file.endswith(".csv"):
                full_path = os.path.join(dirpath, file)
                csv_files.append(full_path)
    return csv_files

# Get list of all CSV files recursively
all_files = find_all_csv_files(root_path)

# Filter out the tracking JSON file
all_files = [f for f in all_files if not f.endswith("processed_files.json")]

# Load list of processed files
try:
    with open(tracking_file, "r") as f:
        processed_files = json.load(f)
except (FileNotFoundError, json.JSONDecodeError):
    processed_files = []

print("Processed files:")
print(processed_files)

# Detect new files
new_files = [f for f in all_files if f not in processed_files]

if new_files:
    print("New files to process:")
    print(new_files)

    # Load all new CSVs
    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .load(new_files)
        .withColumn("ingest_time", current_timestamp())
    )

    # Append to Bronze Delta table
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("0725catalog.bronze.customers_raw")

    # Update tracking file
    processed_files.extend(new_files)
    processed_files = list(set(processed_files))  # optional deduplication
    with open(tracking_file, "w") as f:
        json.dump(processed_files, f)

    print("Tracking file updated.")
else:
    print("No new files to process.")
